# TruthLens — RoBERTa Fine-tuning on LIAR Dataset
**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

In [ ]:
# Install dependencies
!pip install transformers datasets accelerate scikit-learn pandas requests -q

In [ ]:
import os, io, zipfile, requests
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "roberta-base"
OUTPUT_DIR  = "/content/checkpoints/roberta-truthlens"
MAX_LENGTH  = 128
NUM_LABELS  = 2
EPOCHS      = 4
BATCH_SIZE  = 32   # larger batch on GPU

LIAR_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
LABEL_STR_MAP = {
    "pants-fire": 0, "false": 1, "barely-true": 2,
    "half-true": 3, "mostly-true": 4, "true": 5,
}

In [ ]:
# ── Download & load LIAR dataset ──────────────────────────────────────────────
data_dir = "/content/data/liar"
os.makedirs(data_dir, exist_ok=True)

print("Downloading LIAR dataset...")
resp = requests.get(LIAR_URL, timeout=60)
with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall(data_dir)
print("Done.")

def read_split(filename):
    df = pd.read_csv(
        os.path.join(data_dir, filename),
        sep="\t", header=None, usecols=[0,1,2], names=["id","label","statement"]
    )
    df = df[df["label"].isin(LABEL_STR_MAP)]
    df["label"] = df["label"].map(LABEL_STR_MAP)
    # Binary: 0/1/2 → suspicious(1), 3/4/5 → credible(0)
    df["labels"] = df["label"].apply(lambda x: 1 if x in [0,1,2] else 0)
    return Dataset.from_dict({"statement": df["statement"].tolist(), "labels": df["labels"].tolist()})

raw = DatasetDict({
    "train":      read_split("train.tsv"),
    "validation": read_split("valid.tsv"),
    "test":       read_split("test.tsv"),
})
print(raw)

In [ ]:
# ── Tokenize ──────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tokenizer(batch["statement"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    enc["labels"] = batch["labels"]
    return enc

tokenized = raw.map(tokenize, batched=True, remove_columns=["statement"])
tokenized.set_format("torch")
print("Tokenization complete.")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label={0: "credible", 1: "suspicious"},
    label2id={"credible": 0, "suspicious": 1},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "f1":       round(f1_score(labels, preds, average="binary"), 4),
    }

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# ── Evaluate & Save ───────────────────────────────────────────────────────────
results = trainer.evaluate(tokenized["test"])
print(f"Test accuracy: {results['eval_accuracy']}  |  F1: {results['eval_f1']}")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# ── Download checkpoint as zip ────────────────────────────────────────────────
import shutil
shutil.make_archive("/content/roberta-truthlens", "zip", OUTPUT_DIR)
print("Download /content/roberta-truthlens.zip from the Files panel on the left.")